# Plot Neural-EMT-like Tumour Cells on Visium HD Data 

Uses the spatial object from notebook 2 (`adata_sp_final_relabelled.h5ad`; tumour cells labelled
with the snRNA-seq malignant states by scANVI).

1. Location of one tumour state (default `5_Neurod_EMT-like`) on the H&E image of each slide.
2. Candidate genes (from a gene list, e.g. state markers from snRNA-seq) ranked by expression in that
   state **and** by enrichment over the other tumour states.
3. Spatial maps of selected genes within that state, on a colour scale shared across slides.

In [ ]:
# !pip install --quiet scanpy squidpy
# %cd <project>/spatial_visiumHD

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
from scipy import sparse

## Config

In [ ]:
input_path = "adata_sp_final_relabelled.h5ad"

figures_output_dir = "figures/malignant"
tables_output_dir = "tables"

label_col = "cell_label_final"
slide_col = "slide"                      # also the squidpy library_id of each slide
state = "5_Neurod_EMT-like" #5_Neurod_EMT-like ==> represents Neural_EMT-like in manuscript
state_color = "#bc80bd"
# comparison group for enrichment: the other tumour states
tumour_states = ["1_Epithelial baseline", "2_Differentiated epithelial", "3_Devp_stem-like",
                 "4_Inflammatory EMT", "5_Neurod_EMT-like", "6_Proliferative"]

os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)

## 1. Load and normalise (log-normalised copy for plotting)

In [ ]:
adata_sp = sc.read_h5ad(input_path)
vals = adata_sp.X[:1000].data
assert np.allclose(vals, np.round(vals)), "X is expected to contain raw counts"

adata_vis = adata_sp.copy()
sc.pp.normalize_total(adata_vis, target_sum=1e4)
sc.pp.log1p(adata_vis)
slides = sorted(adata_vis.obs[slide_col].unique())
pd.crosstab(adata_vis.obs[label_col], adata_vis.obs[slide_col]).loc[lambda d: d.index.isin(tumour_states)]

## 2. Location of the tumour state on H&E

In [ ]:
for slide in slides:
    sub = adata_sp[(adata_sp.obs[slide_col] == slide) & (adata_sp.obs[label_col] == state)].copy()
    sub.obs[label_col] = pd.Categorical(sub.obs[label_col])
    sub.uns[f"{label_col}_colors"] = [state_color]
    fig, ax = plt.subplots(figsize=(3, 3))
    sq.pl.spatial_scatter(sub, library_id=slide, img=True, img_alpha=0.5, color=label_col, size=state_point_size,
                          legend_loc=None, linewidths=0.1, edgecolor="black", ax=ax)
    ax.set_title(f"{slide} (n = {sub.n_obs})", fontsize=8)
    plt.savefig(f"{figures_output_dir}/{state.split('_', 1)[-1]}_{slide}.pdf", bbox_inches="tight", dpi=150)
    plt.show()

## 3. Candidate genes: expression and enrichment

Mean log-expression and % expressing cells in the state, compared with the other tumour states
(`log2FC` of mean expression, pseudocount 0.01). Ranked by enrichment, then by % expressing, so
genes expressed everywhere (e.g. airway genes) do not rise to the top on expression alone.

In [ ]:
# --- candidate genes ---
gene_file, gene_column = "neurod_like_list.csv", "Gene"
exclude_prefixes = ("MT-",)
n_top = 15

gene_list = pd.read_csv(gene_file)[gene_column].dropna().astype(str).str.strip().unique().tolist()
genes = [g for g in gene_list if g in adata_vis.var_names and not g.startswith(exclude_prefixes)]
print(f"genes in list: {len(gene_list)} | found in spatial data (excl. {exclude_prefixes}): {len(genes)}")


def expr_stats(adata, genes):
    X = adata[:, genes].X
    mean = np.asarray(X.mean(axis=0)).ravel()
    pct = np.asarray((X > 0).mean(axis=0)).ravel() * 100
    return mean, pct


in_state = adata_vis.obs[label_col] == state
others = adata_vis.obs[label_col].isin([s for s in tumour_states if s != state])
mean_in, pct_in = expr_stats(adata_vis[in_state], genes)
mean_out, pct_out = expr_stats(adata_vis[others], genes)

results = pd.DataFrame({"gene": genes, "mean_in_state": mean_in, "pct_in_state": pct_in,
                        "mean_other_tumour": mean_out, "pct_other_tumour": pct_out})
results["log2FC_vs_other_tumour"] = np.log2((results["mean_in_state"] + 0.01) / (results["mean_other_tumour"] + 0.01))
results = results.sort_values(["log2FC_vs_other_tumour", "pct_in_state"], ascending=False).reset_index(drop=True)
results.round(3).to_csv(f"{tables_output_dir}/{state}_candidate_genes_spatial.csv", index=False)
print(f"{in_state.sum()} cells in state, {others.sum()} other tumour cells")
results.head(n_top).round(3)

## 4. Gene maps within the tumour state

All cells of the state are drawn in the state colour; expressing cells are overlaid and coloured
by expression. The colour scale is shared between slides for each gene.

In [ ]:
# --- gene maps ---
genes_to_plot = ["LGR4", "SOX4", "CDH11", "MUC5B", "TENM4", "NAMPT", "TP63", "NGF", "AHR", "NDRG1", "SEMA3A"]
gene_cmap = "Reds"
vmax_percentile = 99                     # colour-scale maximum per gene, shared by both slides
state_point_size, gene_point_size, background_point_size = 40, 25, 12

state_cells = adata_vis[in_state]
for gene in genes_to_plot:
    if gene not in state_cells.var_names:
        print(f"{gene}: not in the spatial data, skipped")
        continue
    expr = state_cells[:, gene].X
    expr = expr.toarray().ravel() if sparse.issparse(expr) else np.asarray(expr).ravel()
    pos = expr > 0
    vmax = np.percentile(expr[pos], vmax_percentile) if pos.any() else 1
    for slide in slides:
        on_slide = (state_cells.obs[slide_col] == slide).to_numpy()
        xy = state_cells.obsm["spatial"]
        fig, ax = plt.subplots(figsize=(3, 3))
        ax.scatter(*xy[on_slide].T, color=state_color, s=background_point_size, linewidths=0, alpha=0.35, zorder=1)
        dots = ax.scatter(*xy[on_slide & pos].T, c=expr[on_slide & pos], cmap=gene_cmap, vmin=0, vmax=vmax,
                          s=gene_point_size, linewidths=0, zorder=2)
        ax.set_aspect("equal")
        ax.invert_yaxis()
        ax.axis("off")
        ax.set_title(f"{gene} | {slide} ({(on_slide & pos).sum()}/{on_slide.sum()} cells)", fontsize=7)
        plt.colorbar(dots, ax=ax, fraction=0.045, pad=0.02).set_label(f"{gene} (log-norm)")
        plt.savefig(f"{figures_output_dir}/{gene}_{slide}_{state.split('_', 1)[-1]}_overlay.pdf", dpi=150, bbox_inches="tight") #Figure 2
        plt.show()